In [ ]:
# ── Cell 1: Mount Drive ──────────────────────────────────────────────
# Run once per session. Grants Colab access to your Google Drive so
# checkpoints and the replay buffer sync automatically.

from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/chess-engine')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('sys.path entry:', PROJECT_ROOT / 'src')

In [ ]:
# ── Cell 2: Install deps ─────────────────────────────────────────────
# Installs from requirements.txt into the Colab runtime.
# Fast on subsequent runs once pip's wheel cache is warm.

!pip install -q -r /content/drive/MyDrive/chess-engine/requirements.txt
print('Dependencies ready.')

In [ ]:
# ── Cell 3: Import ───────────────────────────────────────────────────
# Verify GPU availability and import the training entry point.

import torch
from main import train

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found — training will be slow. Use Runtime > Change runtime type.')

In [ ]:
# ── Cell 4: Train (fresh start) ──────────────────────────────────────
# Runs the full self-play → train → evaluate loop.
# Checkpoints are written to checkpoints/ and data/ every save_every
# iterations, so a Colab disconnect loses at most save_every iters.
#
# If you are resuming after a disconnect, run Cell 5 instead.

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/chess-engine')

train(
    # ── paths ────────────────────────────────────────────────────────
    checkpoints_dir = PROJECT_ROOT / 'checkpoints',
    data_dir        = PROJECT_ROOT / 'data',
    book_path       = PROJECT_ROOT / 'books/gm2001.bin',
    elo_log_path    = PROJECT_ROOT / 'runs/elo_log.csv',
    # ── architecture ─────────────────────────────────────────────────
    num_blocks  = 10,    # residual blocks in the tower
    num_filters = 128,   # conv channels throughout
    # ── self-play ────────────────────────────────────────────────────
    games_per_iter = 25,   # self-play games generated per iteration
    mcts_sims      = 200,  # MCTS simulations per move
    max_game_moves = 512,  # hard cap per game (draw if reached)
    # ── training ─────────────────────────────────────────────────────
    train_steps  = 500,    # gradient updates per iteration
    batch_size   = 256,
    min_buffer   = 1_000,  # wait until buffer has this many examples
    # ── evaluation ───────────────────────────────────────────────────
    eval_every    = 5,     # evaluate every N iterations
    eval_games    = 40,    # games per evaluation match
    eval_sims     = 200,
    win_threshold = 0.55,  # promote if new model wins >= 55 %
    # ── loop control ─────────────────────────────────────────────────
    num_iters  = 1_000,
    save_every = 10,       # checkpoint to Drive every N iterations
)

In [ ]:
# ── Cell 5: Resume ───────────────────────────────────────────────────
# Run this cell after a Colab disconnect or session expiry.
# main.train() reads checkpoints/state.pt automatically and picks up
# from the last completed iteration — no manual bookkeeping needed.
# Keep all hyperparameters identical to the original run.

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/chess-engine')

train(
    # ── paths ────────────────────────────────────────────────────────
    checkpoints_dir = PROJECT_ROOT / 'checkpoints',
    data_dir        = PROJECT_ROOT / 'data',
    book_path       = PROJECT_ROOT / 'books/gm2001.bin',
    elo_log_path    = PROJECT_ROOT / 'runs/elo_log.csv',
    # ── architecture (must match original run) ────────────────────────
    num_blocks  = 10,
    num_filters = 128,
    # ── self-play ────────────────────────────────────────────────────
    games_per_iter = 25,
    mcts_sims      = 200,
    max_game_moves = 512,
    # ── training ─────────────────────────────────────────────────────
    train_steps  = 500,
    batch_size   = 256,
    min_buffer   = 1_000,
    # ── evaluation ───────────────────────────────────────────────────
    eval_every    = 5,
    eval_games    = 40,
    eval_sims     = 200,
    win_threshold = 0.55,
    # ── loop control ─────────────────────────────────────────────────
    num_iters  = 1_000,
    save_every = 10,
)